In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import decoupler as dc
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import warnings
import os
import seaborn as sns
import math

In [ ]:
sc.set_figure_params(dpi_save=300)

# add scpoli labels

In [ ]:
adata = sc.read_h5ad("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/mosaic_dataset/single_cell/preprocessed/sc_merged_annotated.h5ad")

In [ ]:
adata.X = adata.layers["ambient_rna_removed"]
del adata.layers["ambient_rna_removed"]
del adata.layers['LogNormalize']

In [ ]:
adata.obs.rename(columns={'orig.ident': 'sample_id'}, inplace=True)
adata.obs.rename(columns={'celltype_level2_scanvi': 'cluster_id'}, inplace=True)

In [ ]:
adata.X = np.round(adata.X) # only ambientRNA corrected not yet rounded counts were provided

In [ ]:
adata.obs['sample_id'] = adata.obs['sample_id'].str.replace('_', '')
adata.obs['cluster_id'] = adata.obs['cluster_id'].str.replace('_', '')

In [ ]:
adata.write("mofa/mofa_workflow/input_data_mosaic/sc_mosaic_rounded.h5ad")

In [ ]:
adata = sc.read_h5ad("mofa/mofa_workflow/input_data_mosaic/sc_mosaic_rounded.h5ad")

In [ ]:
adata.obs_names = adata.obs_names.str[:-2]

In [ ]:
adata.obs

In [ ]:
df = pd.read_csv("reference_mapping/predictions_with_embeddings.csv") # new labels not subclustered yet, from map_scpoli.ipynb

In [ ]:
df

In [ ]:
# Set the 'barcode' column as the index for easy lookup
df.set_index('barcode', inplace=True)

# Select only the embedding columns
embedding_cols = [col for col in df.columns if col.startswith('embedding_')]
embeddings_df = df[embedding_cols]

# Align the embeddings with adata
adata_barcodes_trimmed = adata.obs_names

# Re-index the embeddings DataFrame to match the exact order and content of adata's cells
aligned_embeddings_df = embeddings_df.reindex(adata_barcodes_trimmed)

# Handle potential mismatches 
missing_count = aligned_embeddings_df.isnull().any(axis=1).sum()
if missing_count > 0:
    print(f"Warning: {missing_count} cells in adata did not have a matching barcode in the CSV.")
    # Fill any missing values with 0. You could also use another strategy.
    aligned_embeddings_df.fillna(0, inplace=True)
    print("Missing values have been filled with 0.")

# Convert to a NumPy array and add to adata.obsm
embeddings_array = aligned_embeddings_df.to_numpy()

# Add the array to the .obsm slot with the key "X_scPoli"
adata.obsm["X_scPoli"] = embeddings_array

print("\n✅ Successfully added embeddings to adata.obsm['X_scPoli']!")

In [ ]:
adata.obs['cluster_id'] = adata.obs.index.map(df['cell_type_pred'])
adata.obs['uncertainty'] = adata.obs.index.map(df['uncertainty'])

In [ ]:
adata[adata.obs['uncertainty']>0.7].obs["cluster_id"].value_counts()

In [ ]:
if 'X_umap' in adata.obsm:
    adata.obsm['X_umap_original'] = np.copy(adata.obsm['X_umap'])

# List of embeddings to compute UMAP for
embeddings = ['X_pca', 'X_scanvi', 'X_scvi', 'X_scPoli']

for emb in tqdm(embeddings):
    # Compute neighbors based on the specific embedding
    sc.pp.neighbors(adata, use_rep=emb)
    
    # Compute UMAP
    sc.tl.umap(adata)
    
    # Save the new UMAP under a descriptive key
    new_key = f'X_umap_{emb[2:]}'  
    adata.obsm[new_key] = np.copy(adata.obsm['X_umap'])

# restore the original X_umap
if 'X_umap_original' in adata.obsm:
    adata.obsm['X_umap'] = np.copy(adata.obsm['X_umap_original'])

In [ ]:
adata.write("mofa/resources/sc_mosaic_rounded_embeds_uncert_nosubclusters.h5ad")

# subclusters from subcluster.ipynb

In [ ]:
adata = sc.read_h5ad("mofa/resources/sc_mosaic_rounded_embeds_uncert_nosubclusters.h5ad")

In [ ]:
adata

In [ ]:
preds1 = pd.read_csv("mofa/E_and_M_MDSCs_newscpoli.csv") # from subcluster.ipynb
preds2 = pd.read_csv("mofa/TRAILAstrocytes_newscpoli.csv") # from subcluster.ipynb

preds = pd.concat([preds1, preds2])
preds = preds.set_index("barcode")
preds

In [ ]:
# preds already has index=barcode and a column 'cell_type_sub'
lookup = preds["cell_type_sub_new"]


# map over your AnnData obs-index (which are barcodes), producing NaN where no match
new_vals = adata.obs.index.to_series().map(lookup)

# fill NaNs with the existing cluster_id, and assign back
adata.obs["cluster_id_trail_mdsc"] = new_vals.fillna(adata.obs["cluster_id"])

In [ ]:
adata.obs['cluster_id_trail_mdsc'].value_counts()

In [ ]:
# remove owkin annots
cols_to_remove = [
    'seurat_clusters', 'seurat_clusters_res_0.1', 'seurat_clusters_res_0.2',
    'seurat_clusters_res_0.4', 'seurat_clusters_res_0.6', 'seurat_clusters_res_0.8',
    'seurat_clusters_res_1', 'seurat_clusters_res_2', 'origin', '_scvi_batch',
    '_scvi_labels', 'scanvi_score', 'celltype_level1_scanvi',
    'celltype_level3_scanvi', 'celltype_level4_scanvi', 'celltype_level_custom1_scanvi',
    'SCT_snn_res.0.4', 'SCT_snn_res.2', 'seurat_clusters_heterogeneity',
    'seurat_clusters_res_1.5', 'seurat_clusters_res_2.5', 'seurat_clusters_res_3',
    'seurat_clusters_res_3.5', 'seurat_clusters_res_4'
]

# Only keep columns that actually exist in the object
cols_to_remove = [col for col in cols_to_remove if col in adata.obs.columns]

adata.obs.drop(columns=cols_to_remove, inplace=True)

In [ ]:
adata.write("mofa/resources/sc_mosaic_rounded_embeds_uncert_subclusters.h5ad") # keep in mind that cells with uncert>0.7 were not included in subclustering

# Plots and analysis

In [ ]:
adata = sc.read_h5ad("mofa/resources/sc_mosaic_rounded_embeds_uncert_subclusters.h5ad") # keep in mind that cells with uncert>0.7 were not included in subclustering

In [ ]:
adata

In [ ]:
adata = adata[adata.obs["uncertainty"]<0.7].copy()

In [ ]:
# only keep baseline samples
baselines = np.array(['HKG001a', 'HKG002a', 'HKG003a', 'HKG004a', 'HKG005a', 'HKG006a',
       'HKG007a', 'HKG008a', 'HKG009a', 'HKG010a', 'HKG011a', 'HKG012a',
       'HKG013a', 'HKG015a', 'HKG016a', 'HKG018a', 'HKG019a', 'HKG020a',
       'HKG021a', 'HKG022a', 'HKG023a', 'HKG024a', 'HKG025a', 'HKG026a',
       'HKG027a', 'HKG028a', 'HKG030a', 'HKG031a', 'HKG032a', 'HKG033a',
       'HKG034a', 'HKG035a', 'HKG037a', 'HKG038a', 'HKG039a', 'HKG040a',
       'HKG041a', 'HKG042a', 'HKG045a', 'HKG046a', 'HKG047a', 'HKG048a',
       'HKG049a', 'HKG050a', 'HKG051a', 'HKG052a', 'HKG053a', 'HKG054a',
       'HKG055a', 'HKG057a', 'HKG058a', 'HKG060a', 'HKG063a', 'HKG064a',
       'HKG065a', 'HKG066a', 'HKG067a', 'HKG068a', 'HKG069a', 'HKG070a',
       'HKG071a', 'HKG072a', 'HKG073a', 'HKG074a', 'HKG075a', 'HKG076a',
       'HKG077a', 'HKG078a', 'HKG080a', 'HKG081a', 'HKG083a', 'HKG086b',
       'HKG087a', 'HKG088a', 'HKG089a', 'HKG092b', 'HKG093a', 'HKG094a',
       'HKG098b', 'HKG099a', 'HKG101a', 'HKG102a', 'HKG103a', 'HKG104a',
       'HKG107a', 'HKG108a', 'HKG110a', 'HKG112a', 'HKG114a'])

In [ ]:
adata = adata[adata.obs['sample_id'].isin(baselines)].copy()

In [ ]:
adata.obs["cluster_id_trail_mdsc"].value_counts()

In [ ]:
color_palette = {
    # Glial/Neural-related cells (Blues and Purples)
    'OPC': '#1f77b4',          # Dark blue
    'OPC-like': '#4b9cd3',     # Lighter blue
    'Oligodendrocyte': '#6baed6',  # Light blue
    'Astrocyte': '#9ecae1',    # Very light blue
    'TRAIL+ Astrocytes': '#ff00ff',  # Bright magenta (distinct from Astrocyte)
    'AC-like': '#6a51a3',      # Dark purple
    'NPC-like': '#8c6bb1',     # Medium purple
    'RG': '#bcbddc',           # Light purple
    'Neuron': '#dadaeb',       # Very light purple

    # Immune cells (Reds, Oranges, Pinks, Browns, Yellows)
    'TAM-MG': '#d62728',       # Dark red
    'TAM-BDM': '#e66767',      # Lighter red
    'E-MDSCs': '#8B4513',      # Saddle brown (unchanged, distinct from oranges)
    'M-MDSCs': '#DAA520',      # Mustard yellow (distinct from brown and oranges)
    'Mono': '#fdae6b',         # Light orange
    'Neutrophil': '#fdd0a2',   # Very light orange
    'DC': '#e31a1c',           # Bright red
    'CD4/CD8': '#fc4e2a',      # Red-orange
    'NK': '#feb24c',           # Yellow-orange
    'B cell': '#ffeda0',       # Pale yellow
    'Plasma B': '#f768a1',     # Pink
    'Mast': '#ae017e',         # Dark pink

    # Vascular/Other (Greens)
    'Endothelial': '#31a354',   # Dark green
    'Mural cell': '#74c476',   # Medium green
    'MES-like': '#bae4b3'      # Light green
}

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

sc.pl.embedding(
    adata,
    color=["cluster_id_trail_mdsc"],
    basis="X_umap_scPoli",
    palette=color_palette,
    legend_loc="right margin",
    ax=ax,
    show=False,
    size=2,
    frameon=False,
    title="Cell Type Annotation"
)

# Calculate centroids for each cell type
cell_types = adata.obs["cluster_id_trail_mdsc"].cat.categories
umap_coords = adata.obsm["X_umap_scPoli"]
centroids = {}

for ct in cell_types:
    mask = adata.obs["cluster_id_trail_mdsc"] == ct
    centroids[ct] = np.mean(umap_coords[mask], axis=0)
    
# manual nudges for the specific overlapping pairs
offsets = {
    "E-MDSC":   (0.6, 0.4),
    "Mono":     (-0.6, -0.4),
    "CD4/CD8":  (0.5, 0.5),
    "NK":       (-0.5, -0.5),
    "OPC-like": (0.5, -0.3),   
    "RG":        (-1.5, 0.3),
}

for ct, (x, y) in centroids.items():
    dx, dy = offsets.get(ct, (0.0, 0.0))
    ax.annotate(
        ct,
        xy=(x, y),                 # point (centroid)
        xytext=(x + dx, y + dy),   # displaced label
        textcoords='data',
        ha="center",
        fontsize=9,
        weight="bold",
        zorder=10,
    )

plt.tight_layout()
plt.savefig("figures/trail_mdsc_scpoli_baselines.pdf")
plt.show()

In [ ]:
sc.pl.embedding(
    adata,
    color=["uncertainty"],
    basis="X_umap_scPoli",
    show=True,
    size=2
)

In [ ]:
cluster_key = "cluster_id_trail_mdsc"
sample_key  = "sample_id"

fig_width_in = 180 / 25.4

tumor_group = ["OPC-like", "NPC-like", "AC-like", "MES-like"]

obs = adata.obs.copy()
obs[cluster_key] = obs[cluster_key].astype(str).str.strip()

present = pd.Index(sorted(obs[cluster_key].unique()))

others = [c for c in present if c not in tumor_group]
tumor_group_present = [c for c in tumor_group if c in present]

# stack order: Tumor Group (Bottom) , Others (Top)
stack_order = tumor_group_present + others 

# Colors
fallback = "#d9d9d9"
cat_colors = {c: color_palette.get(c, fallback) for c in stack_order}

# Global abundances
counts = obs[cluster_key].value_counts(dropna=False).reindex(stack_order).fillna(0).astype(int)
total = int(counts.sum())
props = (counts / total).fillna(0.0)
sorted_idx = props.sort_values(ascending=True).index

# Per-sample stacked data
has_sample = sample_key in obs.columns
if has_sample:
    sample_counts = (
        obs.groupby([sample_key, cluster_key])
           .size()
           .unstack(fill_value=0)
           .reindex(columns=stack_order) 
    )
    sample_props = (sample_counts.T / sample_counts.sum(axis=1).replace(0, np.nan)).T.fillna(0.0)

    # sort samples
    combined_load = sample_props[tumor_group_present].sum(axis=1)
    sample_order = combined_load.sort_values(ascending=False).index.tolist()

fig_height_in = max(3.8, 0.22 * len(sorted_idx))
fig, (axA, axB) = plt.subplots(
    nrows=1, ncols=2,
    figsize=(fig_width_in, fig_height_in),
    dpi=300,
    gridspec_kw={"width_ratios": [1.0, 1.2]}
)

# Left: Global abundance 
vals = props.loc[sorted_idx].values
axA.barh(range(len(sorted_idx)), vals, color=[cat_colors[c] for c in sorted_idx], edgecolor="none")
axA.set_yticks(range(len(sorted_idx)))
axA.set_yticklabels(sorted_idx, fontsize=8)
axA.set_xticks([])
axA.set_title("Global cell type abundance", fontsize=12, pad=8)

for y, c in enumerate(sorted_idx):
    p = props.loc[c]
    x_text = p + 0.004 if p > 0 else 0.004
    axA.text(x_text, y, f"{p*100:.1f}%", va="center", ha="left", fontsize=8)

for spine in ["top", "right", "bottom"]:
    axA.spines[spine].set_visible(False)
axA.set_xlim(0, max(vals.max() * 1.15, 0.06))

# Right: Per-sample stacked bars
if has_sample:
    x = np.arange(len(sample_order))
    bottom = np.zeros(len(sample_order))
    
    # Loop iterates through stack_order (Tumor -> Others)
    for c in stack_order:
        seg = sample_props.loc[sample_order, c].values
        axB.bar(x, seg, bottom=bottom, color=cat_colors[c], width=0.9, edgecolor="none")
        bottom += seg

    axB.set_xticks([])
    axB.set_ylabel("Proportion", fontsize=8)
    axB.set_yticks([0, 0.25, 0.5, 0.75, 1])
    axB.set_yticklabels(["0.0", "0.25", "0.5", "0.75", "1.0"], fontsize=8)
    axB.set_ylim(0, 1)
    axB.set_title("Cell type composition across samples", fontsize=12, pad=8)

    for spine in ["top", "right"]:
        axB.spines[spine].set_visible(False)
else:
    axB.axis("off")

axA.grid(False)
axB.grid(False)
plt.tight_layout()
plt.savefig("figures/celltype_abundances_2col_titles_baselines.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Prepare data (Rank_genes_groups requires log-normalized data)
adata_de = adata.copy()
sc.pp.normalize_total(adata_de, target_sum=1e4)
sc.pp.log1p(adata_de)

In [ ]:
# Calculate Differential Expression (DE)
sc.tl.rank_genes_groups(
    adata_de, 
    groupby='cluster_id_trail_mdsc', 
    method='wilcoxon',
    key_added='rank_genes',
    use_raw=False
)

In [ ]:
adata_de.write("mofa/resources/DE_genes_backup.h5ad")

In [ ]:
adata_de = sc.read_h5ad("mofa/resources/DE_genes_backup.h5ad")

In [ ]:
top_n = 10
result = adata_de.uns['rank_genes']
groups = result['names'].dtype.names
de_markers = {group: list(result['names'][group][:top_n]) for group in groups}
# Get the list of all cluster names 
all_clusters = list(de_markers.keys())

# Split the clusters into 3 roughly equal chunks so it fits in the supplementary figure pdf
cluster_chunks = np.array_split(all_clusters, 3)

# Iterate through chunks and plot
for i, chunk in enumerate(cluster_chunks):
    current_clusters = list(chunk)
    
    subset_markers = {cluster: de_markers[cluster] for cluster in current_clusters}
    
    print(f"Generating plot {i+1}/3 with clusters: {current_clusters}")

    dp = sc.pl.dotplot(
        adata_de, 
        subset_markers, 
        groupby='cluster_id_trail_mdsc',
        standard_scale='var',
        colorbar_title='Mean expression\nin group',
        vmin=0, vmax=1,
        show=False,       
        use_raw=False,
        return_fig=True
    )

    # Style and Save
    dp.style(dot_edge_color='black', dot_edge_lw=1).show()
    
    # Save
    filename = f"figures/top10_de_markers_dotplot_part{i+1}.pdf"
    dp.savefig(filename)
    print(f"Saved: {filename}")
    
    # Close the figure to free memory
    plt.close()

# MOFA prep

In [ ]:
adata = sc.read_h5ad("mofa/resources/sc_mosaic_rounded_embeds_uncert_subclusters.h5ad") # keep in mind that cells with uncert>0.7 were not included in subclustering

In [ ]:
adata = adata[adata.obs["uncertainty"]<0.7].copy()

In [ ]:
adata.obs

In [ ]:
adata.obs.rename(columns={'cluster_id': 'cluster_id_lvl1'}, inplace=True)
adata.obs.rename(columns={'cluster_id_trail_mdsc': 'cluster_id'}, inplace=True)

In [ ]:
adata.write("mofa/resources/sc_mosaic_rounded_embeds_NOuncert_subclusters_mofa.h5ad") # keep in mind that cells with uncert>0.7 were not included in subclustering

# cell type proportions

In [ ]:
adata = sc.read_h5ad("mofa/resources/sc_mosaic_rounded_embeds_NOuncert_subclusters_mofa.h5ad")

In [ ]:
counts = adata.obs.groupby(["sample_id", "cluster_id"]).size().reset_index(name="count")

In [ ]:
counts_wide = counts.pivot(index="sample_id", columns="cluster_id", values="count").fillna(0)

In [ ]:
counts_wide

In [ ]:
counts_wide.to_csv("celltype_counts.csv")

In [ ]:
proportions = pd.read_csv("celltype_props_zComp.csv", index_col=0)

In [ ]:
proportions.index.name = "sample_id"

In [ ]:
proportions

In [ ]:
# Apply the CLR transformation
log_transformed = np.log(proportions)
geometric_mean_log = log_transformed.mean(axis=1)
clr_transformed = log_transformed.subtract(geometric_mean_log, axis=0)
clr_transformed

In [ ]:
clr_transformed.to_csv("celltype_proportions_zComp_CLR.csv")

In [ ]:
clr_transformed.to_csv("mofa/mofa_workflow/input_data_mosaic/celltype_proportions_zComp_CLR.csv")